In [3]:
#!/usr/bin/env Rscript

# ============================================
# U25 consolidated paper outputs (CSV + PDF)
# - Reads per-dataset k-summary CSVs from:
#     .../STEP2_U25/preprocessed_features_OH_U25/
#     .../STEP2_U25/preprocessed_features_FP_U25/
# - Writes consolidated CSVs & PDFs to:
#     .../STEP2_U25/
# - Prints wide table to console
# ============================================

# ---- Safe package loader ----
safe_lib <- function(pkg){
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
    library(pkg, character.only = TRUE)
  }
}
safe_lib("ggplot2"); safe_lib("dplyr"); safe_lib("tidyr"); safe_lib("readr"); safe_lib("stringr"); safe_lib("purrr")

# ---- Paths ----
base_dir <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
root_out <- file.path(base_dir, "20251026_under_25clusters", "STEP2_U25")
today_tag <- format(Sys.Date(), "%Y%m%d")

oh_dir <- file.path(root_out, "preprocessed_features_OH_U25")
fp_dir <- file.path(root_out, "preprocessed_features_FP_U25")

# ---- Utilities ----
find_ksum_file <- function(dir_path){
  if (!dir.exists(dir_path)) return(NA_character_)
  cand <- list.files(dir_path, pattern = "_cluster_k_summary_.*\\.csv$", full.names = TRUE)
  if (length(cand) == 0) return(NA_character_)
  # pick the last modified (most recent)
  cand[which.max(file.info(cand)$mtime)]
}

read_ksum <- function(path, dataset_tag){
  if (is.na(path)) return(NULL)
  df <- tryCatch(readr::read_csv(path, show_col_types = FALSE), error = function(e) NULL)
  if (is.null(df)) return(NULL)
  # Expect columns: mode, index, k   (others will be preserved/ignored as needed)
  if (!all(c("mode","index","k") %in% names(df))) {
    warning("Unexpected columns in ", path, " — expecting at least: mode, index, k")
  }
  df$dataset <- dataset_tag
  df
}

ensure_dir <- function(p){
  if (!dir.exists(p)) dir.create(p, recursive = TRUE, showWarnings = FALSE)
}

# ---- Locate and read input k-summary files ----
oh_ksum_path <- find_ksum_file(oh_dir)
fp_ksum_path <- find_ksum_file(fp_dir)

if (is.na(oh_ksum_path) || is.na(fp_ksum_path)) {
  stop("k-summary CSV not found. Looked for '*_cluster_k_summary_*.csv' in:\n",
       " - ", oh_dir, "\n",
       " - ", fp_dir, "\n",
       "Please run STEP2 to generate per-dataset summaries first.")
}

message("[Found] OH k-summary: ", oh_ksum_path)
message("[Found] FP k-summary: ", fp_ksum_path)

oh_ksum <- read_ksum(oh_ksum_path, "OH")
fp_ksum <- read_ksum(fp_ksum_path, "FP")

if (is.null(oh_ksum) || is.null(fp_ksum)) {
  stop("Failed to read one or both k-summary CSVs.")
}

ksum_all <- dplyr::bind_rows(oh_ksum, fp_ksum) %>%
  dplyr::mutate(
    mode    = factor(mode,  levels = c("top3","cumeig")),
    index   = factor(index, levels = c("silhouette","dunn","gap","ch","db","ptbiserial")),
    dataset = factor(dataset, levels = c("OH","FP"))
  )

# ---- Consolidated CSVs (under STEP2_U25) ----
ensure_dir(root_out)

out_csv_long <- file.path(root_out, sprintf("U25_k_summary_all_long_%s.csv", today_tag))
readr::write_csv(ksum_all, out_csv_long)
message("[Saved] ", out_csv_long)

ksum_wide <- ksum_all %>%
  dplyr::select(dataset, mode, index, k) %>%
  tidyr::pivot_wider(names_from = c(dataset, mode),
                     values_from = k,
                     names_sep   = "_") %>%
  dplyr::arrange(index)

out_csv_wide <- file.path(root_out, sprintf("U25_k_summary_all_wide_%s.csv", today_tag))
readr::write_csv(ksum_wide, out_csv_wide)
message("[Saved] ", out_csv_wide)

# ---- Console preview ----
message("\n--- Console preview: U25_k_summary_all_wide ---")
print(ksum_wide, n = nrow(ksum_wide))

# ---- Figures (under STEP2_U25) ----
# Figure 1: Heatmap of chosen k (index × mode), facets by dataset
fig_heat <- ggplot2::ggplot(
    ksum_all %>% dplyr::filter(!is.na(k)),
    ggplot2::aes(x = mode, y = index, fill = k)
  ) +
  ggplot2::geom_tile() +
  ggplot2::geom_text(ggplot2::aes(label = k)) +
  ggplot2::facet_wrap(~ dataset, nrow = 1) +
  ggplot2::labs(
    title = "Chosen number of clusters (k) by index and dimension mode",
    x = "Dimension mode", y = "Index", fill = "k"
  ) +
  ggplot2::theme_minimal(base_size = 12) +
  ggplot2::theme(
    plot.title = ggplot2::element_text(face = "bold"),
    axis.text.x = ggplot2::element_text(angle = 0, vjust = 0.5)
  )

out_pdf_heat <- file.path(root_out, sprintf("Fig_U25_k_heatmap_%s.pdf", today_tag))
# Try cairo for better text rendering; fall back if cairo is unavailable
ok <- TRUE
tryCatch({
  ggplot2::ggsave(out_pdf_heat, fig_heat, width = 9, height = 4.8, device = cairo_pdf)
}, error = function(e){ ok <<- FALSE })
if (!ok) ggplot2::ggsave(out_pdf_heat, fig_heat, width = 9, height = 4.8)

message("[Saved] ", out_pdf_heat)

# Figure 2: Bar chart (k by index), facets dataset × mode
fig_bar <- ggplot2::ggplot(
    ksum_all %>% dplyr::filter(!is.na(k)),
    ggplot2::aes(x = index, y = k)
  ) +
  ggplot2::geom_col(width = 0.7) +
  ggplot2::facet_grid(dataset ~ mode) +
  ggplot2::labs(
    title = "Chosen number of clusters (k) by index",
    x = "Index", y = "k"
  ) +
  ggplot2::theme_minimal(base_size = 12) +
  ggplot2::theme(
    plot.title = ggplot2::element_text(face = "bold"),
    axis.text.x = ggplot2::element_text(angle = 25, hjust = 1)
  )

out_pdf_bar <- file.path(root_out, sprintf("Fig_U25_k_bar_%s.pdf", today_tag))
ok2 <- TRUE
tryCatch({
  ggplot2::ggsave(out_pdf_bar, fig_bar, width = 10, height = 6, device = cairo_pdf)
}, error = function(e){ ok2 <<- FALSE })
if (!ok2) ggplot2::ggsave(out_pdf_bar, fig_bar, width = 10, height = 6)

message("[Saved] ", out_pdf_bar)

message("\n✅ Done. Consolidated CSVs and PDFs are under:\n", root_out)


Loading required package: ggplot2

Loading required package: dplyr


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Loading required package: tidyr

Loading required package: readr

Loading required package: stringr

Loading required package: purrr

[Found] OH k-summary: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/preprocessed_features_OH_U25/preprocessed_features_OH_U25_cluster_k_summary_20251026.csv

[Found] FP k-summary: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/preprocessed_features_FP_U25/preprocessed_features_FP_U25_cluster_k_summary_20251026.csv

[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/U25_k_summary_all_long_20251026.csv

[Saved] /Volumes/csbdeep11/_yasu-i/20250

# A tibble: 6 x 5
  index      OH_top3 OH_cumeig FP_top3 FP_cumeig
  <fct>        <dbl>     <dbl>   <dbl>     <dbl>
1 silhouette       2         2      23        25
2 dunn             2        14       5         2
3 gap              2         2       2         2
4 ch              25         2      25        25
5 db               3         3       3         3
6 ptbiserial       5        25       6         6


[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/Fig_U25_k_heatmap_20251026.pdf

[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/Fig_U25_k_bar_20251026.pdf


<U+2705> Done. Consolidated CSVs and PDFs are under:
/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25



In [5]:
#!/usr/bin/env Rscript

# ============================================
# U25 consolidated paper outputs (CSV + PDF) + Index Profiles (k=2..25)
# - Reads per-dataset k-summary CSVs from:
#     .../STEP2_U25/preprocessed_features_OH_U25/
#     .../STEP2_U25/preprocessed_features_FP_U25/
# - Recomputes MDS from original features (corr -> 1-corr -> cmdscale)
#   and builds index profiles (x=k, y=score) for:
#     silhouette, dunn, gap, ch, db, ptbiserial  over k=2..25
# - Saves index profiles to each dataset folder:
#     index_profile_{mode}_{index}.csv
# - Writes consolidated CSVs & PDFs to:
#     .../STEP2_U25/   and   .../STEP2_U25/profiles/
# - Prints wide table + chosen/best-k summary to console
# ============================================

# ---- Safe package loader ----
safe_lib <- function(pkg){
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg, repos = "https://cloud.r-project.org")
    library(pkg, character.only = TRUE)
  }
}
safe_lib("ggplot2"); safe_lib("dplyr"); safe_lib("tidyr"); safe_lib("readr"); safe_lib("stringr"); safe_lib("purrr")
safe_lib("cluster");   # silhouette, clusGap
safe_lib("fpc");       # calinhara (CH), cluster.stats (Dunn)


# ---- Paths ----
base_dir <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"
root_out <- file.path(base_dir, "20251026_under_25clusters", "STEP2_U25")
profiles_out <- file.path(root_out, "profiles")
today_tag <- format(Sys.Date(), "%Y%m%d")

oh_dir <- file.path(root_out, "preprocessed_features_OH_U25")
fp_dir <- file.path(root_out, "preprocessed_features_FP_U25")

# ---- Utilities (I/O) ----
ensure_dir <- function(p){
  if (!dir.exists(p)) dir.create(p, recursive = TRUE, showWarnings = FALSE)
}
find_ksum_file <- function(dir_path){
  if (!dir.exists(dir_path)) return(NA_character_)
  cand <- list.files(dir_path, pattern = "_cluster_k_summary_.*\\.csv$", full.names = TRUE)
  if (length(cand) == 0) return(NA_character_)
  cand[which.max(file.info(cand)$mtime)]  # most recent
}
read_ksum <- function(path, dataset_tag){
  if (is.na(path)) return(NULL)
  df <- tryCatch(readr::read_csv(path, show_col_types = FALSE), error = function(e) NULL)
  if (is.null(df)) return(NULL)
  if (!all(c("mode","index","k") %in% names(df))) {
    warning("Unexpected columns in ", path, " — expecting at least: mode, index, k")
  }
  df$dataset <- dataset_tag
  df
}

# ---- Utilities (features -> MDS coords) ----
# ---- Indices (pure R implementations) ----
# Davies–Bouldin index (lower is better)
compute_db_index <- function(X, cl){
  # X: matrix/data.frame (rows=samples), cl: integer vector of cluster labels
  if (length(unique(cl)) < 2) return(NA_real_)
  X <- as.matrix(X)
  labs <- sort(unique(cl))
  k <- length(labs)

  # centroids
  cent <- sapply(labs, function(L){
    colMeans(X[cl == L, , drop = FALSE])
  })
  cent <- t(cent) # k x p

  # scatter within cluster: mean distance to centroid
  S <- numeric(k)
  for (i in seq_len(k)){
    idx <- which(cl == labs[i])
    if (length(idx) <= 1){
      S[i] <- 0  # singleton → 0 とする
    } else {
      ci <- cent[i, , drop = FALSE]
      S[i] <- mean(sqrt(rowSums((X[idx, , drop = FALSE] - matrix(ci, nrow=length(idx), ncol=ncol(X), byrow=TRUE))^2)))
    }
  }

  # centroid distances
  Dc <- as.matrix(dist(cent))
  R <- matrix(NA_real_, k, k)
  for (i in seq_len(k)){
    for (j in seq_len(k)){
      if (i == j) next
      if (Dc[i, j] == 0) {
        R[i, j] <- Inf  # 代表点が同一 → 最悪ケース
      } else {
        R[i, j] <- (S[i] + S[j]) / Dc[i, j]
      }
    }
  }
  # DB = mean_i max_{j!=i} R_ij
  db <- mean(apply(R, 1, function(v) max(v[is.finite(v)], na.rm = TRUE)), na.rm = TRUE)
  if (!is.finite(db)) return(NA_real_)
  db
}

# Point-biserial index (higher is better)
# 定義：距離ベクトル d_ij と「同一クラスタか」の2値ベクトル s_ij の相関の負符号（-cor）
# ＝同一クラスタほど距離が小さい → 相関は負になるので、-cor で「大きいほど良い」にする
compute_ptbiserial_index <- function(D, cl){
  # D: dist オブジェクト（上三角）, cl: クラスタラベル
  n <- attr(D, "Size")
  if (length(unique(cl)) < 2 || n < 2) return(NA_real_)
  # 上三角の (i<j) ペアを列挙して s を作る
  s <- logical(n * (n - 1) / 2)
  idx <- 1
  for (i in 1:(n-1)){
    for (j in (i+1):n){
      s[idx] <- (cl[i] == cl[j])
      idx <- idx + 1
    }
  }
  d <- as.numeric(D)
  if (length(unique(s)) < 2) return(NA_real_)  # 全部同一/全部異なる等
  r <- suppressWarnings(cor(d, as.numeric(s), method = "pearson", use = "complete.obs"))
  if (is.na(r)) return(NA_real_)
  -r  # 大きいほど良い（同クラスタ=1 で距離が小→負相関 → 符号反転）
}

                   
read_numeric_matrix <- function(path){
  df <- read.delim(path, header = TRUE, sep = ",", row.names = 1, as.is = TRUE, strip.white = FALSE)
  is_num <- !vapply(df, is.character, logical(1))
  if (!any(is_num)) stop("No numeric columns: ", path)
  df[, is_num, drop = FALSE]
}
compute_mds_from_corr <- function(numData, k_max = 50){
  corData <- suppressWarnings(cor(numData, use = "pairwise.complete.obs"))
  corData[is.na(corData)] <- 0
  ddata <- dist(1 - corData)
  k <- min(k_max, max(1, ncol(corData) - 1))
  mds <- cmdscale(ddata, k = k, eig = TRUE)
  eig_vals <- mds$eig
  totaleig <- sum(abs(eig_vals))
  peigen   <- if (totaleig == 0) rep(0, length(eig_vals)) else abs(eig_vals) / totaleig
  contrib <- data.frame(
    axis        = seq_along(eig_vals),
    eigenvalue  = eig_vals,
    percent     = peigen,
    cum_percent = cumsum(peigen),
    stringsAsFactors = FALSE
  )
  list(mds = mds, contrib = contrib)
}
select_dims <- function(contrib_df, mode, avail_dims){
  pos_idx <- which(contrib_df$percent > 0)
  if (length(pos_idx) == 0) return(min(3, max(1, avail_dims)))
  if (mode == "top3"){
    k <- min(3, length(pos_idx))
  } else {
    thr <- 0.8
    k <- which(contrib_df$cum_percent[pos_idx] >= thr)[1]
    if (is.na(k)) k <- length(pos_idx)
  }
  k <- min(k, avail_dims); k <- max(1, k)
  k
}

# ---- Build MDS coord matrix X for a dataset/mode ----
get_X_for <- function(dataset_tag = c("OH","FP"), mode = c("top3","cumeig")){
  dataset_tag <- match.arg(dataset_tag); mode <- match.arg(mode)
  in_csv <- file.path(base_dir, if (dataset_tag=="OH") "preprocessed_features_OH.csv" else "preprocessed_features_FP.csv")
  numData <- read_numeric_matrix(in_csv)
  m <- compute_mds_from_corr(numData)
  coords <- as.matrix(m$mds$points)
  if (is.null(dim(coords))) { coords <- matrix(coords, ncol = 1); colnames(coords) <- "Dim1" }
  k_dim <- select_dims(m$contrib, mode, ncol(coords))
  X <- coords[, 1:k_dim, drop = FALSE]
  X
}

# ---- Compute and save index profiles for k=2..25 ----
# Helper: ward.D2 clustering function for clusGap
ward_cut <- function(x, k){
  cl <- cutree(hclust(dist(x), method = "ward.D2"), k)
  list(cluster = cl)
}
# Robust scorer w/ tryCatch returning NA on failure
safe_score <- function(expr){
  out <- NA_real_
  try({ out <- eval(expr) }, silent = TRUE)
  out
}
compute_index_profiles <- function(X, mode_tag, out_dir, k_min = 2, k_max = 25, gap_B = 20){
  ensure_dir(out_dir)
  n <- nrow(X)
  k_max <- min(k_max, max(2, n-1))

  # GAP (once per mode), uses clusGap which computes all k in one shot
  gap_tab <- NULL
  if (n >= 3) {
    gap_obj <- try(cluster::clusGap(X, FUNcluster = ward_cut, K.max = k_max, B = gap_B, verbose = FALSE), silent = TRUE)
    if (!inherits(gap_obj, "try-error")) {
      gap_tab <- as.data.frame(gap_obj$Tab)
      gap_tab$k <- seq_len(nrow(gap_tab))
      gap_tab <- gap_tab[gap_tab$k >= k_min & gap_tab$k <= k_max, c("k","gap")]
      names(gap_tab) <- c("k","score")
      readr::write_csv(gap_tab, file.path(out_dir, sprintf("index_profile_%s_gap.csv", mode_tag)))
      message("[Saved index profile] ", file.path(out_dir, sprintf("index_profile_%s_gap.csv", mode_tag)))
    }
  }

  ks <- k_min:k_max
  # Preallocate containers for other indices
  sil  <- dunn <- db <- ch <- ptb <- numeric(length(ks)); sil[] <- dunn[] <- db[] <- ch[] <- ptb[] <- NA_real_

  D <- dist(X)

  for (i in seq_along(ks)){
    k <- ks[i]
    cl <- cutree(hclust(D, method = "ward.D2"), k)

    # silhouette (mean)
    sil[i] <- safe_score( quote( mean(cluster::silhouette(cl, D)[,3]) ) )
    # Dunn (higher better) via fpc::cluster.stats
    dstat  <- try(fpc::cluster.stats(D, cl), silent = TRUE)
    if (!inherits(dstat, "try-error")) dunn[i] <- dstat$dunn

    # Davies–Bouldin (lower better)
    db[i] <- safe_score( quote( compute_db_index(X, cl) ) )


    # Calinski–Harabasz (higher better)
    ch[i] <- safe_score( quote( fpc::calinhara(X, cl, cn = max(cl)) ) )

    # Point-biserial (higher better)
    ptb[i] <- safe_score( quote( compute_ptbiserial_index(D, cl) ) )
  }

  # Save per-index CSVs
  readr::write_csv(data.frame(k = ks, score = sil), file.path(out_dir, sprintf("index_profile_%s_silhouette.csv", mode_tag)))
  message("[Saved index profile] ", file.path(out_dir, sprintf("index_profile_%s_silhouette.csv", mode_tag)))

  readr::write_csv(data.frame(k = ks, score = dunn), file.path(out_dir, sprintf("index_profile_%s_dunn.csv", mode_tag)))
  message("[Saved index profile] ", file.path(out_dir, sprintf("index_profile_%s_dunn.csv", mode_tag)))

  if (!is.null(gap_tab)) {
    # already saved in GAP section
  } else {
    # in case clusGap failed, still create an empty stub to avoid missing file confusion
    readr::write_csv(data.frame(k = ks, score = NA_real_), file.path(out_dir, sprintf("index_profile_%s_gap.csv", mode_tag)))
    message("[Saved index profile] (stub) ", file.path(out_dir, sprintf("index_profile_%s_gap.csv", mode_tag)))
  }

  readr::write_csv(data.frame(k = ks, score = ch), file.path(out_dir, sprintf("index_profile_%s_ch.csv", mode_tag)))
  message("[Saved index profile] ", file.path(out_dir, sprintf("index_profile_%s_ch.csv", mode_tag)))

  readr::write_csv(data.frame(k = ks, score = db), file.path(out_dir, sprintf("index_profile_%s_db.csv", mode_tag)))
  message("[Saved index profile] ", file.path(out_dir, sprintf("index_profile_%s_db.csv", mode_tag)))

  readr::write_csv(data.frame(k = ks, score = ptb), file.path(out_dir, sprintf("index_profile_%s_ptbiserial.csv", mode_tag)))
  message("[Saved index profile] ", file.path(out_dir, sprintf("index_profile_%s_ptbiserial.csv", mode_tag)))
}

# ============================================
# 1) Read chosen-k summaries
# ============================================
oh_ksum_path <- find_ksum_file(oh_dir)
fp_ksum_path <- find_ksum_file(fp_dir)
if (is.na(oh_ksum_path) || is.na(fp_ksum_path)) {
  stop("k-summary CSV not found. Looked for '*_cluster_k_summary_*.csv' in:\n",
       " - ", oh_dir, "\n",
       " - ", fp_dir, "\n",
       "Please run STEP2 to generate per-dataset summaries first.")
}
message("[Found] OH k-summary: ", oh_ksum_path)
message("[Found] FP k-summary: ", fp_ksum_path)

oh_ksum <- read_ksum(oh_ksum_path, "OH")
fp_ksum <- read_ksum(fp_ksum_path, "FP")
if (is.null(oh_ksum) || is.null(fp_ksum)) stop("Failed to read one or both k-summary CSVs.")

indices <- c("silhouette","dunn","gap","ch","db","ptbiserial")
modes   <- c("top3","cumeig")

ksum_all <- dplyr::bind_rows(oh_ksum, fp_ksum) %>%
  dplyr::mutate(
    mode    = factor(mode,  levels = modes),
    index   = factor(index, levels = indices),
    dataset = factor(dataset, levels = c("OH","FP"))
  )

# ============================================
# 2) (NEW) Compute & save index profiles per dataset × mode
#     - Recompute MDS coords from preprocessed_features_{OH,FP}.csv
# ============================================
message("\n=== Building index profiles (k=2..25) ===")
# OH
X_oh_top3   <- get_X_for("OH", "top3");   compute_index_profiles(X_oh_top3,   "top3",   oh_dir)
X_oh_cumeig <- get_X_for("OH", "cumeig"); compute_index_profiles(X_oh_cumeig, "cumeig", oh_dir)
# FP
X_fp_top3   <- get_X_for("FP", "top3");   compute_index_profiles(X_fp_top3,   "top3",   fp_dir)
X_fp_cumeig <- get_X_for("FP", "cumeig"); compute_index_profiles(X_fp_cumeig, "cumeig", fp_dir)

# ============================================
# 3) Consolidated CSVs (under STEP2_U25)
# ============================================
ensure_dir(root_out); ensure_dir(profiles_out)

out_csv_long <- file.path(root_out, sprintf("U25_k_summary_all_long_%s.csv", today_tag))
readr::write_csv(ksum_all, out_csv_long)
message("[Saved] ", out_csv_long)

ksum_wide <- ksum_all %>%
  dplyr::select(dataset, mode, index, k) %>%
  tidyr::pivot_wider(names_from = c(dataset, mode),
                     values_from = k,
                     names_sep   = "_") %>%
  dplyr::arrange(index)

out_csv_wide <- file.path(root_out, sprintf("U25_k_summary_all_wide_%s.csv", today_tag))
readr::write_csv(ksum_wide, out_csv_wide)
message("[Saved] ", out_csv_wide)

# ---- Console preview ----
message("\n--- Console preview: U25_k_summary_all_wide ---")
print(ksum_wide, n = nrow(ksum_wide))

# ============================================
# 4) Figures (under STEP2_U25): heatmap + bar
# ============================================
fig_heat <- ggplot2::ggplot(
    ksum_all %>% dplyr::filter(!is.na(k)),
    ggplot2::aes(x = mode, y = index, fill = k)
  ) +
  ggplot2::geom_tile() +
  ggplot2::geom_text(ggplot2::aes(label = k)) +
  ggplot2::facet_wrap(~ dataset, nrow = 1) +
  ggplot2::labs(
    title = "Chosen number of clusters (k) by index and dimension mode",
    x = "Dimension mode", y = "Index", fill = "k"
  ) +
  ggplot2::theme_minimal(base_size = 12) +
  ggplot2::theme(
    plot.title = ggplot2::element_text(face = "bold"),
    axis.text.x = ggplot2::element_text(angle = 0, vjust = 0.5)
  )
out_pdf_heat <- file.path(root_out, sprintf("Fig_U25_k_heatmap_%s.pdf", today_tag))
ok <- TRUE
tryCatch({ ggplot2::ggsave(out_pdf_heat, fig_heat, width = 9, height = 4.8, device = cairo_pdf) },
         error = function(e){ ok <<- FALSE })
if (!ok) ggplot2::ggsave(out_pdf_heat, fig_heat, width = 9, height = 4.8)
message("[Saved] ", out_pdf_heat)

fig_bar <- ggplot2::ggplot(
    ksum_all %>% dplyr::filter(!is.na(k)),
    ggplot2::aes(x = index, y = k)
  ) +
  ggplot2::geom_col(width = 0.7) +
  ggplot2::facet_grid(dataset ~ mode) +
  ggplot2::labs(
    title = "Chosen number of clusters (k) by index",
    x = "Index", y = "k"
  ) +
  ggplot2::theme_minimal(base_size = 12) +
  ggplot2::theme(
    plot.title = ggplot2::element_text(face = "bold"),
    axis.text.x = ggplot2::element_text(angle = 25, hjust = 1)
  )
out_pdf_bar <- file.path(root_out, sprintf("Fig_U25_k_bar_%s.pdf", today_tag))
ok2 <- TRUE
tryCatch({ ggplot2::ggsave(out_pdf_bar, fig_bar, width = 10, height = 6, device = cairo_pdf) },
         error = function(e){ ok2 <<- FALSE })
if (!ok2) ggplot2::ggsave(out_pdf_bar, fig_bar, width = 10, height = 6)
message("[Saved] ", out_pdf_bar)

# ============================================
# 5) Index Profile Figures (per dataset × mode)
# ============================================
# loader for profiles
load_index_profile <- function(dir_path, mode, index_name){
  f <- file.path(dir_path, sprintf("index_profile_%s_%s.csv", mode, index_name))
  if (!file.exists(f)) return(NULL)
  df <- tryCatch(readr::read_csv(f, show_col_types = FALSE), error = function(e) NULL)
  if (is.null(df) || !all(c("k","score") %in% names(df))) return(NULL)
  df[order(df$k), , drop = FALSE]
}
best_k_from_profile <- function(df, index_name){
  if (is.null(df) || nrow(df) == 0) return(c(NA_integer_, NA_real_))
  if (tolower(index_name) %in% c("db")) {
    i <- which.min(df$score)
  } else {
    i <- which.max(df$score)
  }
  c(as.integer(df$k[i]), as.numeric(df$score[i]))
}
build_profiles_one <- function(dset_tag, dset_dir, mode_tag){
  indices <- c("silhouette","dunn","gap","ch","db","ptbiserial")
  vk <- ksum_all %>% dplyr::filter(dataset == dset_tag, mode == mode_tag) %>% dplyr::select(index, k)

  # console chosen vs best
  message(sprintf("\n--- Console: %s / %s index profiles ---", dset_tag, mode_tag))
  for (ix in indices){
    prof <- load_index_profile(dset_dir, mode_tag, ix)
    chosen_k <- vk$k[vk$index == ix]
    chosen_k <- ifelse(length(chosen_k)==0, NA_integer_, chosen_k)
    if (!is.null(prof)){
      bk <- best_k_from_profile(prof, ix)
      message(sprintf("%s/%s/%s: best k*=%s, score*=%s | chosen k=%s",
                      dset_tag, mode_tag, ix,
                      ifelse(is.na(bk[1]), "NA", as.character(bk[1])),
                      ifelse(is.na(bk[2]), "NA", sprintf("%.4g", bk[2])),
                      ifelse(is.na(chosen_k), "NA", as.character(chosen_k))))
    } else {
      message(sprintf("%s/%s/%s: (no profile) | chosen k=%s",
                      dset_tag, mode_tag, ix,
                      ifelse(is.na(chosen_k), "NA", as.character(chosen_k))))
    }
  }

  # build figure
  ensure_dir(profiles_out)
  out_pdf <- file.path(profiles_out, sprintf("Fig_U25_index_profiles_%s_%s_%s.pdf", dset_tag, mode_tag, today_tag))

  # collect all profiles stacked
  prof_all <- purrr::map_dfr(indices, function(ix){
    df <- load_index_profile(dset_dir, mode_tag, ix)
    if (is.null(df)) return(NULL)
    df$index <- factor(ix, levels = indices)
    df
  })

  if (!is.null(prof_all) && nrow(prof_all) > 0){
    vline_df <- vk %>% dplyr::mutate(index = factor(index, levels = indices))
    p <- ggplot2::ggplot(prof_all, ggplot2::aes(k, score)) +
      ggplot2::geom_line() +
      ggplot2::geom_vline(data = vline_df, ggplot2::aes(xintercept = k), linetype = "dashed") +
      ggplot2::facet_wrap(~ index, ncol = 3, scales = "free_y") +
      ggplot2::labs(title = sprintf("Index profiles — %s — %s", dset_tag, mode_tag),
                    x = "Clusters (k)", y = "Index score") +
      ggplot2::theme_minimal(base_size = 12) +
      ggplot2::theme(plot.title = ggplot2::element_text(face = "bold"))
  } else {
    # empty with vlines if any chosen_k exist
    dummy <- data.frame(index = factor(indices, levels = indices), k = NA_real_, score = NA_real_)
    vline_df <- vk %>% dplyr::mutate(index = factor(index, levels = indices))
    p <- ggplot2::ggplot(dummy, ggplot2::aes(k, score)) +
      ggplot2::geom_blank() +
      ggplot2::geom_vline(data = vline_df, ggplot2::aes(xintercept = k), linetype = "dashed") +
      ggplot2::facet_wrap(~ index, ncol = 3) +
      ggplot2::annotate("text", x = 0.5, y = 0.5, label = "no profile",
                        vjust = 0.5, hjust = 0.5, size = 4, colour = "grey40", alpha = 0.9) +
      ggplot2::labs(title = sprintf("Index profiles — %s — %s (no profiles found)", dset_tag, mode_tag),
                    x = "Clusters (k)", y = "Index score") +
      ggplot2::theme_minimal(base_size = 12) +
      ggplot2::theme(plot.title = ggplot2::element_text(face = "bold"))
  }

  okp <- TRUE
  tryCatch({ ggplot2::ggsave(out_pdf, p, width = 12, height = 7, device = cairo_pdf) },
           error = function(e){ okp <<- FALSE })
  if (!okp) ggplot2::ggsave(out_pdf, p, width = 12, height = 7)
  message("[Saved] ", out_pdf)
}

# Build profile figures (now that CSVs exist)
build_profiles_one("OH", oh_dir, "top3")
build_profiles_one("OH", oh_dir, "cumeig")
build_profiles_one("FP", fp_dir, "top3")
build_profiles_one("FP", fp_dir, "cumeig")

message("\n✅ Done. Consolidated CSVs and PDFs are under:\n", root_out, "\nProfiles under:\n", profiles_out)


[Found] OH k-summary: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/preprocessed_features_OH_U25/preprocessed_features_OH_U25_cluster_k_summary_20251026.csv

[Found] FP k-summary: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/preprocessed_features_FP_U25/preprocessed_features_FP_U25_cluster_k_summary_20251026.csv


=== Building index profiles (k=2..25) ===

[Saved index profile] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/preprocessed_features_OH_U25/index_profile_top3_gap.csv

[Saved index profile] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/preprocessed_features_OH_U25/index_profile_top3_silhouette.csv

[Saved index profile] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/preprocessed_features_OH_U25/index_profile_top3_dunn.csv

[Saved 

# A tibble: 6 x 5
  index      OH_top3 OH_cumeig FP_top3 FP_cumeig
  <fct>        <dbl>     <dbl>   <dbl>     <dbl>
1 silhouette       2         2      23        25
2 dunn             2        14       5         2
3 gap              2         2       2         2
4 ch              25         2      25        25
5 db               3         3       3         3
6 ptbiserial       5        25       6         6


[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/Fig_U25_k_heatmap_20251026.pdf

[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/Fig_U25_k_bar_20251026.pdf


--- Console: OH / top3 index profiles ---

OH/top3/silhouette: best k*=NA, score*=NA | chosen k=2

OH/top3/dunn: best k*=2, score*=0.1532 | chosen k=2

OH/top3/gap: best k*=25, score*=2.169 | chosen k=2

OH/top3/ch: best k*=2, score*=146.6 | chosen k=25

OH/top3/db: best k*=2, score*=0.7893 | chosen k=3

OH/top3/ptbiserial: best k*=NA, score*=NA | chosen k=5

Warning message:
"Removed 48 rows containing missing values or values outside the scale range
(`geom_line()`)."
[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters/STEP2_U25/profiles/Fig_U25_index_profiles_OH_top3_20251026.pdf


--- Console: OH / cumeig index profiles ---

OH/cumeig/silhouette: best k*=NA, score*=NA | chosen 